In [0]:
# Step 1: Get the catalog name

catalog = dbutils.widgets.get("catalog")
schema = "bharat_bricks"
table_name = "gold_posts_chunked"
full_table = f"{catalog}.{schema}.{table_name}"

print(f"Creating {full_table} with Change Data Feed enabled...")

Creating iitb.bharat_bricks.gold_posts_chunked with Change Data Feed enabled...


In [0]:
# Step 2: Create gold_posts_chunked as a Delta table with CDF enabled

sql_query = f"""
WITH comments_agg AS (
  -- Aggregate comments per post in chronological order
  SELECT
    post_id,
    ARRAY_JOIN(
      TRANSFORM(
        ARRAY_SORT(
          COLLECT_LIST(
            named_struct('ts', created_at, 'text', CONCAT('[', author, ' | score: ', CAST(score AS STRING), '] ', body))
          )
        ),
        x -> x.text
      ),
      '\\n\\n'
    ) AS comments_text
  FROM {catalog}.{schema}.gold_comments
  GROUP BY post_id
),
combined AS (
  -- Combine post content with aggregated comments into a single document
  SELECT
    p.post_id,
    p.title,
    p.author,
    p.created_at,
    p.flair,
    p.score,
    p.upvote_ratio,
    p.num_comments,
    p.content_type,
    p.permalink,
    CONCAT(
      p.title, '\\n\\n',
      COALESCE(p.body, ''), '\\n\\n',
      'Comments:\\n\\n',
      COALESCE(ca.comments_text, 'No comments yet.')
    ) AS full_text
  FROM {catalog}.{schema}.gold_posts p
  LEFT JOIN comments_agg ca ON p.post_id = ca.post_id
),
chunked AS (
  -- Split full_text at paragraph/comment boundaries (\\n\\n), accumulating
  -- segments into chunks of ~4000 chars without truncating any segment
  SELECT
    c.*,
    AGGREGATE(
      SPLIT(c.full_text, '\\n\\n'),
      named_struct('chunks', FILTER(ARRAY(''), x -> FALSE), 'cur', ''),
      (acc, seg) ->
        IF(LENGTH(CONCAT(acc.cur, '\\n\\n', seg)) > 4000 AND acc.cur != '',
           named_struct('chunks', array_append(acc.chunks, acc.cur), 'cur', seg),
           named_struct('chunks', acc.chunks, 'cur',
             IF(acc.cur = '', seg, CONCAT(acc.cur, '\\n\\n', seg)))
        ),
      acc -> FILTER(array_append(acc.chunks, acc.cur), x -> LENGTH(x) > 0)
    ) AS chunks_array
  FROM combined c
)
-- Explode chunks into individual rows for vector search embedding
SELECT
  post_id,
  title,
  author,
  created_at,
  flair,
  score,
  upvote_ratio,
  num_comments,
  content_type,
  permalink,
  chunk_index,
  CONCAT(post_id, '_', CAST(chunk_index AS STRING)) AS chunk_id,
  chunk_text,
  LENGTH(chunk_text) AS chunk_text_length,
  SIZE(chunks_array) AS total_chunks
FROM chunked
LATERAL VIEW POSEXPLODE(chunks_array) t AS chunk_index, chunk_text
"""

# Create the DataFrame
df = spark.sql(sql_query)

# Write as Delta table with Change Data Feed enabled
df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("delta.enableChangeDataFeed", "true") \
  .option("overwriteSchema", "true") \
  .option("comment", "Chunked r/iitbombay posts with chronological comments for vector search. Chunks split at comment/paragraph boundaries, targeting ~4000 chars without truncating any comment.") \
  .saveAsTable(full_table)